<div style="position: relative;" class="flex flex-col"><div class="HTMLContent_html__0OZLp" data-track-load="description_content"><p>You are given an array of <code>k</code> linked-lists <code>lists</code>, each linked-list is sorted in ascending order.</p>

<p><em>Merge all the linked-lists into one sorted linked-list and return it.</em></p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>

<pre><strong>Input:</strong> lists = [[1,4,5],[1,3,4],[2,6]]
<strong>Output:</strong> [1,1,2,3,4,4,5,6]
<strong>Explanation:</strong> The linked-lists are:
[
  1-&gt;4-&gt;5,
  1-&gt;3-&gt;4,
  2-&gt;6
]
merging them into one sorted linked list:
1-&gt;1-&gt;2-&gt;3-&gt;4-&gt;4-&gt;5-&gt;6
</pre>

<p><strong class="example">Example 2:</strong></p>

<pre><strong>Input:</strong> lists = []
<strong>Output:</strong> []
</pre>

<p><strong class="example">Example 3:</strong></p>

<pre><strong>Input:</strong> lists = [[]]
<strong>Output:</strong> []
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>k == lists.length</code></li>
	<li><code>0 &lt;= k &lt;= 10<sup>4</sup></code></li>
	<li><code>0 &lt;= lists[i].length &lt;= 500</code></li>
	<li><code>-10<sup>4</sup> &lt;= lists[i][j] &lt;= 10<sup>4</sup></code></li>
	<li><code>lists[i]</code> is sorted in <strong>ascending order</strong>.</li>
	<li>The sum of <code>lists[i].length</code> will not exceed <code>10<sup>4</sup></code>.</li>
</ul>
</div><span style="font-size: 0px; line-height: 0;">&nbsp;</span></div>

## Initial intuition

I can see that the size of the items in the linked lists do not cover a really big range (from -10000 to +10000). I believe that a counting sort could work here, where we count how many occurences of each item there are, and reconstruct the linked list at the end. However, this feels like a waste of the fact that the lists are already in order. 

### Topics

Since I was curious in how to utilise the order, I decided to take a look at the topics to get some sort of hint, which are Linked List (helpful), Divide and Conquer, Heap (Priority Queue) and Merge Sort. This gave me an idea in creating a min heap from the linked lists, running the heapify operation after adding each item from each linked list, as this operation is O(logn) time. I am not too confident on this solution however, so i decided to stick with the counting sort

### Initial attempt at solution

Since counting sort starts at 0 (since it uses an array), I will add 10000 to the values when adding to the count sort, and then subtract them by 10000 when creating the Linked List 

In [ ]:
from typing import List, Optional

# Definition for singly-linked list.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        countSort = [0 for i in range(20001)]

        for i in range(len(lists)):
            curList = lists[i]
            while curList != None:
                countSort[curList.val + 10000] += 1
                curList = curList.next

        curNode = ListNode()
        head = curNode

        for i in range(len(countSort)):
            while countSort[i] != 0:
                tempNode = ListNode(i - 10000, None)
                countSort[i] -= 1
                curNode.next = tempNode
                curNode = curNode.next

        return head.next
                


## Result

This worked out. However, the solution only beat 11%, and it still did not make use of the fact that the lists were already sorted. So, I wanted to take a look at the leetcode solution now, so that I can understand it and implement it for myself (unfortunately, it is locked, so I used the top comments solution, thanks niits: https://leetcode.com/problems/merge-k-sorted-lists/solutions/5425987/video-merge-two-lists-at-a-time-by-niits-45i1/).

### Improved solution using merge sort

From the video, I gathered that what should be done is similar to the final stages of merge sort. We have already sorted subarrays (or sublists in this case) that we simply need to merge together.

![Merge sory by Wikipedia](https://upload.wikimedia.org/wikipedia/commons/thumb/e/e6/Merge_sort_algorithm_diagram.svg/500px-Merge_sort_algorithm_diagram.svg.png) 

So it would be like putting together the green blocks together. With this knowledge, I decided to reattempt the question.

In [ ]:
from typing import List, Optional

# Definition for singly-linked list.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Solution1:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        if len(lists) == 0:
            return None

        if len(lists) == 1:
            return lists[0]
        
        tempList = [ListNode() for _ in range((len(lists) // 2) + (len(lists) % 2))]

        while len(lists) > 1:
            for i in range(0, len(lists), 2):
                if i + 1 < len(lists): #Checking for two to merge
                    tempList[int(i/2)] = self.mergeLists(lists[i], lists[i+1])
                else:
                    tempList[i//2] = lists[i]
            
            lists = tempList

            tempList = [ListNode() for _ in range((len(lists) // 2) + (len(lists) % 2))]

        return lists[0]


    def mergeLists(self, list1: ListNode, list2: ListNode):
        result = ListNode()
        head = result

        while list1 != None and list2 != None:
            if list1.val < list2.val:
                result.next = ListNode(list1.val, None)
                list1 = list1.next
            else:
                result.next = ListNode(list2.val, None)
                list2 = list2.next
            result = result.next

        while list1 != None:
            result.next = ListNode(list1.val, None)
            list1 = list1.next
            result = result.next

        while list2 != None:
            result.next = ListNode(list2.val, None)
            list2 = list2.next
            result = result.next

        return head.next

            

                


In [28]:
def arrToList(arr):
    head = ListNode()
    result = head

    for i in arr:
        result.next = ListNode(i, None)
        result = result.next

    return head.next


s = Solution1()
x = s.mergeKLists([arrToList([1,4,5]),arrToList([1,3,4]),arrToList([2,6])])

while x != None:
    print(x.val)
    x = x.next

0
1
1
1
2
3
4
4
5
6
